In [ ]:
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"


Prepare paths of input images and target segmentation masks

In [ ]:
import os
import PIL
from PIL.ImageFilter import MedianFilter
import numpy as np

input_dir_sim = "../output_data/ML-AI/recons-1472"
target_dir_sim = "../output_data/ML-AI/labels-1472"

input_dir_real = "../../data/Wire-Cu-2mm-17.54umvx/CIL-recons-cil"
target_dir_real = "../../data/Wire-Cu-2mm-17.54umvx/manual-segmentation"

img_size = (64, 64)
num_classes = 3
batch_size = 16


In [ ]:
input_img_paths = sorted(
    [
        os.path.join(input_dir_real, fname)
        for fname in os.listdir(input_dir_real)
        if fname.endswith(".png")
    ]
)
target_img_paths = sorted(
    [
        os.path.join(target_dir_real, fname)
        for fname in os.listdir(target_dir_real)
        if fname.endswith(".png") and not fname.startswith(".")
    ]
)

input_img_paths += sorted(
    [
        os.path.join(input_dir_sim, fname)
        for fname in os.listdir(input_dir_sim)
        if fname.endswith(".png")
    ]
)
target_img_paths += sorted(
    [
        os.path.join(target_dir_sim, fname)
        for fname in os.listdir(target_dir_sim)
        if fname.endswith(".png") and not fname.startswith(".")
    ]
)

print("Number of samples:", len(input_img_paths))

for input_path, target_path in zip(input_img_paths, target_img_paths):
    print(input_path, "|", target_path)


In [ ]:
# import os
#
# input_dir = "images/"
# target_dir = "annotations/trimaps/"
# img_size = (160, 160)
# num_classes = 3
# batch_size = 32
#
# input_img_paths = sorted(
#     [
#         os.path.join(input_dir, fname)
#         for fname in os.listdir(input_dir)
#         if fname.endswith(".jpg")
#     ]
# )
# target_img_paths = sorted(
#     [
#         os.path.join(target_dir, fname)
#         for fname in os.listdir(target_dir)
#         if fname.endswith(".png") and not fname.startswith(".")
#     ]
# )
#
# print("Number of samples:", len(input_img_paths))
#
# for input_path, target_path in zip(input_img_paths[:10], target_img_paths[:10]):
#     print(input_path, "|", target_path)


What does one input image and corresponding segmentation mask look like?

In [ ]:
from IPython.display import Image, display
from keras.utils import load_img
from PIL import ImageOps
#
# # Display input image #7
# display(Image(filename=input_img_paths[9]))
#
# # Display auto-contrast version of corresponding target (per-pixel categories)
# img = ImageOps.autocontrast(load_img(target_img_paths[9]))
# display(img)

Prepare dataset to load & vectorize batches of data

In [ ]:
import keras
import numpy as np
from tensorflow import data as tf_data
from tensorflow import image as tf_image
from tensorflow import io as tf_io
import tensorflow as tf

def get_dataset(
    batch_size,
    img_size,
    input_img_paths,
    target_img_paths,
    max_dataset_len=None,
):
    """Returns a TF Dataset."""

    def load_img_masks(input_img_path, target_img_path):
        input_img = tf_io.read_file(input_img_path)
        input_img = tf_io.decode_png(input_img, channels=1)
        input_img = tf_image.resize(input_img, img_size)
        input_img = tf_image.grayscale_to_rgb(input_img)
        input_img = tf_image.convert_image_dtype(input_img, "float32")
        input_img /= np.iinfo('uint16').max

        if target_img_paths:
            target_img = tf_io.read_file(target_img_path)
            target_img = tf_io.decode_png(target_img, channels=1)
            target_img = tf_image.resize(target_img, img_size, method="nearest")
            target_img = tf_image.convert_image_dtype(target_img, "uint8")
        else:
            target_img = None

        # Ground truth labels are 1, 2, 3. Subtract one to make them 0, 1, 2:
        # target_img -= 1
        return input_img, target_img

    # For faster debugging, limit the size of data
    if max_dataset_len:
        input_img_paths = input_img_paths[:max_dataset_len]

    if target_img_paths:
        target_img_paths = target_img_paths[:max_dataset_len]

    dataset = tf_data.Dataset.from_tensor_slices((input_img_paths, target_img_paths))

    dataset = dataset.map(load_img_masks, num_parallel_calls=tf_data.AUTOTUNE)

    return dataset.batch(batch_size)


Prepare U-Net Xception-style model

In [ ]:
from keras import layers

# See https://github.com/qubvel/segmentation_models/tree/master
import segmentation_models as sm

def get_model(img_size, num_classes):
    inputs = keras.Input(shape=img_size + (3,))

    # Don't forget to rescale input images to the [0–1] range.
    # x = layers.Rescaling(1.0 / np.iinfo('uint16').max)(inputs)

    ### [First half of the network: downsampling inputs] ###

    # Entry block
    x = layers.Conv2D(32, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    previous_block_activation = x  # Set aside residual

    # Blocks 1, 2, 3 are identical apart from the feature depth.
    for filters in [64, 128, 256]:
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

        # Project residual
        residual = layers.Conv2D(filters, 1, strides=2, padding="same")(
            previous_block_activation
        )
        x = layers.add([x, residual])  # Add back residual
        previous_block_activation = x  # Set aside next residual

    ### [Second half of the network: upsampling inputs] ###

    for filters in [256, 128, 64, 32]:
        x = layers.Activation("relu")(x)
        x = layers.Conv2DTranspose(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.Conv2DTranspose(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.UpSampling2D(2)(x)

        # Project residual
        residual = layers.UpSampling2D(2)(previous_block_activation)
        residual = layers.Conv2D(filters, 1, padding="same")(residual)
        x = layers.add([x, residual])  # Add back residual
        previous_block_activation = x  # Set aside next residual

    # Add a per-pixel classification layer
    outputs = layers.Conv2D(num_classes, 3, activation="softmax", padding="same")(x)

    # Define the model
    model = keras.Model(inputs, outputs)
    return model

# def get_model(img_size, num_classes):
#     inputs = keras.Input(shape=img_size + (3,))
#
#     # Don't forget to rescale input images to the [0–1] range.
#     x = layers.Rescaling(1.0 / np.iinfo('uint16').max)(inputs)
#
#     # We use padding="same" everywhere to avoid the influence of border
#     # padding on feature map size.
#     x = layers.Conv2D(64, 3, strides=2, activation="relu", padding="same")(x)
#     x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
#     x = layers.Conv2D(128, 3, strides=2, activation="relu", padding="same")(x)
#     x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
#     x = layers.Conv2D(256, 3, strides=2, padding="same", activation="relu")(x)
#     x = layers.Conv2D(256, 3, activation="relu", padding="same")(x)
#
#     x = layers.Conv2DTranspose(256, 3, activation="relu", padding="same")(x)
#     x = layers.Conv2DTranspose(256, 3, strides=2, activation="relu", padding="same")(x)
#     x = layers.Conv2DTranspose(128, 3, activation="relu", padding="same")(x)
#     x = layers.Conv2DTranspose(128, 3, strides=2, activation="relu", padding="same")(x)
#     x = layers.Conv2DTranspose(64, 3, activation="relu", padding="same")(x)
#     x = layers.Conv2DTranspose(64, 3, strides=2, activation="relu", padding="same")(x)
#
#     # We end the model with a per-pixel three-way softmax to classify
#     # each output pixel into one of our three categories.
#     outputs = layers.Conv2D(num_classes, 3, activation="softmax", padding="same")(x)
#
#     return keras.Model(inputs, outputs)

# Build model
# model = get_model(img_size, num_classes)

BACKBONE = 'resnet34'
model = sm.Unet(BACKBONE, classes=num_classes, activation='sigmoid')
# model = sm.Unet(BACKBONE, classes=num_classes, encoder_weights='imagenet')
model.summary()



# model = tf.keras.models.Sequential([
#     layers.Flatten(input_shape=[256, 256, 3]),
#     layers.Dense(64, activation='relu'),
#     layers.Dense(256*256*2, activation='softmax'),
#     layers.Reshape((256, 256, 2))
# ])

Set aside a validation split

In [ ]:
import random

# Split our img paths into a training and a validation set
split = 80
val_samples = round((100 - split) / 100 * len(input_img_paths))
random.Random(1337).shuffle(input_img_paths)
random.Random(1337).shuffle(target_img_paths)
train_input_img_paths = input_img_paths[:-val_samples]
train_target_img_paths = target_img_paths[:-val_samples]
val_input_img_paths = input_img_paths[-val_samples:]
val_target_img_paths = target_img_paths[-val_samples:]

# Instantiate dataset for each split
# Limit input files in `max_dataset_len` for faster epoch training time.
# Remove the `max_dataset_len` arg when running with full dataset.
train_dataset = get_dataset(
    batch_size, img_size, train_input_img_paths,train_target_img_paths
)

valid_dataset = get_dataset(
    batch_size, img_size, val_input_img_paths, val_target_img_paths
)


In [ ]:
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
])

In [ ]:
background_iou = keras.metrics.IoU(
    # Specifies the total number of classes
    num_classes=3,
    # Specifies the class to compute IoU for (0 = background)
    target_class_ids=(0,),
    name="background_iou",
    # Our targets are sparse (integer class IDs).
    sparse_y_true=True,
    # But our model's predictions are a dense softmax!
    sparse_y_pred=False,
)

insulation_iou = keras.metrics.IoU(
    # Specifies the total number of classes
    num_classes=3,
    # Specifies the class to compute IoU for (1 = insulation)
    target_class_ids=(1,),
    name="insulation_iou",
    # Our targets are sparse (integer class IDs).
    sparse_y_true=True,
    # But our model's predictions are a dense softmax!
    sparse_y_pred=False,
)

copper_iou = keras.metrics.IoU(
    # Specifies the total number of classes
    num_classes=3,
    # Specifies the class to compute IoU for (2 = copper)
    target_class_ids=(2,),
    name="copper_iou",
    # Our targets are sparse (integer class IDs).
    sparse_y_true=True,
    # But our model's predictions are a dense softmax!
    sparse_y_pred=False,
)


Train the model

In [ ]:
# Configure the model for training.
# We use the "sparse" version of categorical_crossentropy
# because our target data is integers.
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[sm.metrics.iou_score, background_iou, insulation_iou, copper_iou],
)

# model.compile(
#     optimizer="adam",
#     loss=sm.losses.bce_jaccard_loss,
#     metrics=[sm.metrics.iou_score, background_iou, insulation_iou, copper_iou],
# )


output_path = "../../notebooks/output_data/ML-AI"

csv_logger = keras.callbacks.CSVLogger(os.path.join(output_path, "history.csv"), append=True, separator=";")
model_fname = os.path.join(output_path,
                           "unet-" + BACKBONE + "-segmentation-" + str(img_size[0]) + "x" + str(img_size[1]) + "-" +
                           str(batch_size) + ".keras")

callbacks = [
    keras.callbacks.ModelCheckpoint(
        model_fname,
        save_best_only=True,
        monitor='val_loss',
        mode='min',
    ),
    csv_logger,
]
# Train the model, doing validation at the end of each epoch.
epochs = 50
history = model.fit(
    train_dataset,
    epochs=epochs,
    validation_data=valid_dataset,
    callbacks=callbacks,
    batch_size = batch_size,
    verbose=2,
    # class_weight = {0: 1.0, 1:2.0, 2:3.0}
)


Display the training and validation loss

In [ ]:
from matplotlib import pyplot as plt

epochs = range(1, len(history.history["loss"]) + 1)
loss = history.history["loss"]
val_loss = history.history["val_loss"]

# accuracy = history.history["accuracy"]
# val_accuracy = history.history["val_accuracy"]

plt.figure()
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.plot(epochs, loss, "r--", label="Training loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

# plt.figure()
# plt.plot(epochs, accuracy, "r--", label="Training accuracy")
# plt.plot(epochs, val_accuracy, "b", label="Validation accuracy")
# plt.title("Training and validation accuracy")
# plt.legend()
# plt.show()


Visualize predictions

In [ ]:
# Generate predictions for all images in the validation set

val_dataset = get_dataset(
    batch_size, img_size, val_input_img_paths, val_target_img_paths
)
val_preds = model.predict(val_dataset)


def display_mask(i):
    """Quick utility to display a model's prediction."""
    mask = np.argmax(val_preds[i], axis=-1)
    mask = np.expand_dims(mask, axis=-1)
    img = ImageOps.autocontrast(keras.utils.array_to_img(mask))
    display(img)


# Display results for last validation image
i = -1

# Display input image
display(Image(filename=val_input_img_paths[i]))

# Display ground-truth target mask
img = ImageOps.autocontrast(load_img(val_target_img_paths[i]))
display(img)

# Display mask predicted by our model
display_mask(i)  # Note that the model only sees inputs at 150x150.


In [ ]:
model = keras.models.load_model(model_fname)

In [ ]:
from tifffile import imread
import PIL

CT_image = imread("/run/media/fpvidal/DATA/CT/2023/DTHE/Wire-Cu-2mm-17.54umvx/flexible/slice_idx_0483.tiff")
min_value = np.min(CT_image)
max_value = np.max(CT_image)
value_range = max_value - min_value

uint8_data = np.iinfo('uint8').max * (CT_image.astype(np.single) - min_value) / value_range
uint16_data = np.iinfo('uint16').max * (CT_image.astype(np.single) - min_value) / value_range

PIL_image_uint8  = PIL.Image.fromarray(uint8_data.astype(np.uint8))
PIL_image_uint16 = PIL.Image.fromarray(uint16_data.astype(np.uint16))

PIL_image_uint8.save(os.path.join(output_path, "slice_uint8_idx_0483.png"))
PIL_image_uint16.save(os.path.join(output_path, "slice_uint16_idx_0483.png"))

In [ ]:
dataset = get_dataset(
    batch_size, img_size, [os.path.join(output_path, "slice_uint16_idx_0483.png")], None
)
preds = model.predict(dataset)




In [ ]:
plt.imshow(PIL.Image.open(os.path.join(output_path, "slice_uint16_idx_0483.png")), cmap="gray")
plt.show()

plt.imshow(np.array(preds[0,:,:,0]), cmap="gray")
plt.show()

plt.imshow(np.array(preds[0,:,:,1]), cmap="gray")
plt.show()

plt.imshow(np.array(preds[0,:,:,2]), cmap="gray")
plt.show()
# PIL_image_mask  = PIL.Image.fromarray(np.array(preds[0]))
# PIL_image_mask.save(os.path.join(output_path, "mask_idx_0483.png"))